In [1]:
import requests
import pandas as pd
from datetime import datetime
import os

API_KEY = input("Enter your API key: ")
URL = "https://api.gamebrain.co/v1/games?query=medieval+strategy+games"


def fetch_data():
    headers = {"x-api-key": API_KEY}
    response = requests.get(URL, headers=headers)
    response.raise_for_status()
    return response.json()


def transform(data):
    results = data["results"]

    rows = []
    for game in results:
        rows.append({
            "game_id": game.get("id"),
            "name": game.get("name"),
            "year": game.get("year"),
            "genre": game.get("genre"),
            "rating_mean": game.get("rating", {}).get("mean"),
            "rating_count": game.get("rating", {}).get("count"),
            "adult_only": game.get("adult_only"),
            "query": data.get("query"),
        })

    df = pd.DataFrame(rows)

    # Add ingestion metadata
    today = datetime.today()
    df["ingestion_date"] = today.date()
    df["year_partition"] = 2025
    df["month_partition"] = 1

    return df


def save_parquet(df):
    base_path = "data/games/year=2025/month=01"
    os.makedirs(base_path, exist_ok=True)

    file_path = f"{base_path}/games_2025_01.parquet"
    df.to_parquet(file_path, index=False)

    print(f"Saved: {file_path}")


if __name__ == "__main__":
    data = fetch_data()
    df = transform(data)
    save_parquet(df)

ModuleNotFoundError: No module named 'requests'